# Recent Websocket Capture Files

Inspect the newest compressed JSONL websocket files written by `modl btc`.

The notebook reads samples through the `zstd` CLI, so it can usually peek at files that are still being appended. For active files, the newest buffered records may not be visible until the recorder flushes or rolls the file.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path(os.environ.get("MODL_WS_RAW_DIR", "/mnt/burner-archive/ws_raw"))
RECENT_LIMIT = 40
SAMPLE_ROWS = 5

pd.set_option("display.max_colwidth", 160)
ROOT

PosixPath('/mnt/burner-archive/ws_raw')

In [2]:
def capture_parts(path: Path, root: Path = ROOT) -> dict[str, str | None]:
    rel = path.relative_to(root)
    parts = rel.parts
    if len(parts) < 4:
        return {"exchange": None, "symbol": None, "channel": None}
    return {"exchange": parts[0], "symbol": parts[1], "channel": parts[2]}


def recent_capture_files(root: Path = ROOT, limit: int = RECENT_LIMIT) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*.jsonl.zst"):
        stat = path.stat()
        rows.append(
            {
                **capture_parts(path, root),
                "modified_at": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc),
                "size_mb": stat.st_size / 1024 / 1024,
                "path": str(path),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["exchange", "symbol", "channel", "modified_at", "size_mb", "path"])
    return pd.DataFrame(rows).sort_values("modified_at", ascending=False).head(limit).reset_index(drop=True)


recent = recent_capture_files()
recent

,exchange,symbol,channel,modified_at,size_mb,path
0,extended,btc-usd,orderbook,2026-06-29 22:06:49.057580+00:00,6.794048,/mnt/burner-archive/ws_raw/extended/btc-usd/orderbook/btc-usd_orderbook_26-06-29.jsonl.zst
1,bitfinex,tbtcusd,book_l25,2026-06-29 22:06:45.457546+00:00,1.913250,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-29.jsonl.zst
2,hibachi,btc_usdt-p,market_data,2026-06-29 22:06:40.167497+00:00,2.823638,/mnt/burner-archive/ws_raw/hibachi/btc_usdt-p/market_data/btc_usdt-p_market_data_26-06-29.jsonl.zst
3,extended,btc-usd,mark_price,2026-06-29 22:05:16.640057+00:00,0.120283,/mnt/burner-archive/ws_raw/extended/btc-usd/mark_price/btc-usd_mark_price_26-06-29.jsonl.zst
4,extended,btc-usd,index_price,2026-06-29 22:04:46.337007+00:00,0.140517,/mnt/burner-archive/ws_raw/extended/btc-usd/index_price/btc-usd_index_price_26-06-29.jsonl.zst
5,hyperliquid,ubtc_usdc,trades,2026-06-29 22:02:47.335338+00:00,0.034681,/mnt/burner-archive/ws_raw/hyperliquid/ubtc_usdc/trades/ubtc_usdc_trades_26-06-29.jsonl.zst
6,hyperliquid,ubtc_usdc,book,2026-06-29 22:02:40.971946+00:00,0.085130,/mnt/burner-archive/ws_raw/hyperliquid/ubtc_usdc/book/ubtc_usdc_book_26-06-29.jsonl.zst
7,extended,btc-usd,trades,2026-06-29 21:59:10.063317+00:00,0.104139,/mnt/burner-archive/ws_raw/extended/btc-usd/trades/btc-usd_trades_26-06-29.jsonl.zst
8,extended,btcspot-usd,trades,2026-06-29 21:46:39.829576+00:00,0.001279,/mnt/burner-archive/ws_raw/extended/btcspot-usd/trades/btcspot-usd_trades_26-06-29.jsonl.zst
9,extended,btcspot-usd,orderbook,2026-06-29 21:46:39.829576+00:00,0.015449,/mnt/burner-archive/ws_raw/extended/btcspot-usd/orderbook/btcspot-usd_orderbook_26-06-29.jsonl.zst


In [3]:
def read_jsonl_zst_head(path: str | Path, rows: int = SAMPLE_ROWS) -> pd.DataFrame:
    path = Path(path)
    if shutil.which("zstd") is None:
        raise RuntimeError("The zstd CLI is required. Install zstd or run this notebook on the recorder host.")

    proc = subprocess.Popen(
        ["zstd", "-dc", str(path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    parsed = []
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            line = line.strip()
            if not line:
                continue
            parsed.append(json.loads(line))
            if len(parsed) >= rows:
                break
    finally:
        proc.kill()
        proc.wait(timeout=5)

    return pd.DataFrame(parsed)


def expand_payload_text(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "payload_text" not in df.columns:
        return df
    out = df.copy()
    payloads = []
    for value in out["payload_text"]:
        if not isinstance(value, str):
            payloads.append(None)
            continue
        try:
            payloads.append(json.loads(value))
        except json.JSONDecodeError:
            payloads.append(value)
    out["payload"] = payloads
    return out


if recent.empty:
    raise RuntimeError(f"No .jsonl.zst files found under {ROOT}")

newest_path = recent.loc[0, "path"]
sample = expand_payload_text(read_jsonl_zst_head(newest_path, SAMPLE_ROWS))
newest_path, sample

('/mnt/burner-archive/ws_raw/extended/btc-usd/orderbook/btc-usd_orderbook_26-06-29.jsonl.zst',
                 received_at   received_mts  exchange   symbol    channel  \
 0  2026-06-29T19:39:06.130Z  1782761946130  extended  BTC-USD  orderbook   
 1  2026-06-29T19:39:06.141Z  1782761946141  extended  BTC-USD  orderbook   
 2  2026-06-29T19:39:06.141Z  1782761946141  extended  BTC-USD  orderbook   
 3  2026-06-29T19:39:06.141Z  1782761946141  extended  BTC-USD  orderbook   
 4  2026-06-29T19:39:06.141Z  1782761946141  extended  BTC-USD  orderbook   
 
                 connection_id  \
 0  extended-BTC-USD-orderbook   
 1  extended-BTC-USD-orderbook   
 2  extended-BTC-USD-orderbook   
 3  extended-BTC-USD-orderbook   
 4  extended-BTC-USD-orderbook   
 
                                                                                                                                                       payload_text  \
 0  {"type":"SNAPSHOT","data":{"t":"SNAPSHOT","m":"BTC-USD","b":[{"q

In [4]:
for row in recent.head(10).itertuples(index=False):
    print(f"\n{row.exchange}/{row.symbol}/{row.channel}  {row.size_mb:.3f} MB")
    display(expand_payload_text(read_jsonl_zst_head(row.path, min(3, SAMPLE_ROWS))))


extended/btc-usd/orderbook  6.794 MB


,received_at,received_mts,exchange,symbol,channel,connection_id,payload_text,payload
0,2026-06-29T19:39:06.130Z,1782761946130,extended,BTC-USD,orderbook,extended-BTC-USD-orderbook,"{""type"":""SNAPSHOT"",""data"":{""t"":""SNAPSHOT"",""m"":""BTC-USD"",""b"":[{""q"":""5.63715"",""p"":""60274""},{""q"":""0.49782"",""p"":""60273""},{""q"":""1.19110"",""p"":""60272""},{""q"":""0.683...","{'type': 'SNAPSHOT', 'data': {'t': 'SNAPSHOT', 'm': 'BTC-USD', 'b': [{'q': '5.63715', 'p': '60274'}, {'q': '0.49782', 'p': '60273'}, {'q': '1.19110', 'p': '..."
1,2026-06-29T19:39:06.141Z,1782761946141,extended,BTC-USD,orderbook,extended-BTC-USD-orderbook,"{""type"":""DELTA"",""data"":{""t"":""DELTA"",""m"":""BTC-USD"",""b"":[{""q"":""-0.09568"",""p"":""60274"",""c"":""5.54147""},{""q"":""0.00018"",""p"":""60271"",""c"":""0.68331""},{""q"":""0.00062"",""...","{'type': 'DELTA', 'data': {'t': 'DELTA', 'm': 'BTC-USD', 'b': [{'q': '-0.09568', 'p': '60274', 'c': '5.54147'}, {'q': '0.00018', 'p': '60271', 'c': '0.68331..."
2,2026-06-29T19:39:06.141Z,1782761946141,extended,BTC-USD,orderbook,extended-BTC-USD-orderbook,"{""type"":""DELTA"",""data"":{""t"":""DELTA"",""m"":""BTC-USD"",""b"":[{""q"":""-5.05589"",""p"":""60274"",""c"":""0.48558""},{""q"":""-0.00018"",""p"":""60271"",""c"":""0.68313""},{""q"":""0.00094"",...","{'type': 'DELTA', 'data': {'t': 'DELTA', 'm': 'BTC-USD', 'b': [{'q': '-5.05589', 'p': '60274', 'c': '0.48558'}, {'q': '-0.00018', 'p': '60271', 'c': '0.6831..."



bitfinex/tbtcusd/book_l25  1.913 MB


,received_at,received_mts,exchange,symbol,channel,connection_id,payload_text,payload
0,2026-06-29T19:39:04.911Z,1782761944911,bitfinex,tBTCUSD,book_l25,bitfinex-tBTCUSD-book_l25,"{""event"":""info"",""version"":2,""serverId"":""521b8767-ba15-4141-aae9-573d56db436d"",""platform"":{""status"":1}}","{'event': 'info', 'version': 2, 'serverId': '521b8767-ba15-4141-aae9-573d56db436d', 'platform': {'status': 1}}"
1,2026-06-29T19:39:07.429Z,1782761947429,bitfinex,tBTCUSD,book_l25,bitfinex-tBTCUSD-book_l25,"{""event"":""subscribed"",""channel"":""book"",""chanId"":1372681,""symbol"":""tBTCUSD"",""prec"":""P0"",""freq"":""F0"",""len"":""25"",""pair"":""BTCUSD""}","{'event': 'subscribed', 'channel': 'book', 'chanId': 1372681, 'symbol': 'tBTCUSD', 'prec': 'P0', 'freq': 'F0', 'len': '25', 'pair': 'BTCUSD'}"
2,2026-06-29T19:39:07.429Z,1782761947429,bitfinex,tBTCUSD,book_l25,bitfinex-tBTCUSD-book_l25,"[1372681,[[60360,1,0.08283631],[60359,1,0.08283631],[60358,2,0.20223615],[60356,1,0.08284171],[60350,1,0.0002],[60349,1,0.08284592],[60346,1,0.0002],[60344,...","[1372681, [[60360, 1, 0.08283631], [60359, 1, 0.08283631], [60358, 2, 0.20223615], [60356, 1, 0.08284171], [60350, 1, 0.0002], [60349, 1, 0.08284592], [6034..."



hibachi/btc_usdt-p/market_data  2.824 MB


...

,received_at,received_mts,exchange,symbol,channel,connection_id,payload_text,payload
0,2026-06-29T19:39:55.812Z,1782761995812,extended,BTC-USD,trades,extended-BTC-USD-trades,"{""data"":[{""i"":2071679117232705552,""m"":""BTC-USD"",""S"":""SELL"",""tT"":""TRADE"",""T"":1782761766142,""p"":""60217"",""q"":""0.00011""},{""i"":2071679117232705553,""m"":""BTC-USD"",...","{'data': [{'i': 2071679117232705552, 'm': 'BTC-USD', 'S': 'SELL', 'tT': 'TRADE', 'T': 1782761766142, 'p': '60217', 'q': '0.00011'}, {'i': 207167911723270555..."
1,2026-06-29T19:39:57.797Z,1782761997797,extended,BTC-USD,trades,extended-BTC-USD-trades,"{""data"":[{""i"":2071680087828205573,""m"":""BTC-USD"",""S"":""BUY"",""tT"":""TRADE"",""T"":1782761997550,""p"":""60256"",""q"":""0.06650""}],""ts"":1782761997557,""seq"":2}","{'data': [{'i': 2071680087828205573, 'm': 'BTC-USD', 'S': 'BUY', 'tT': 'TRADE', 'T': 1782761997550, 'p': '60256', 'q': '0.06650'}], 'ts': 1782761997557, 'se..."
2,2026-06-29T19:40:30.448Z,1782762030448,extended,BTC-USD,trades,extended-BTC-USD-trades,"{""data"":[{""i"":2071680224830951424,""m"":""BTC-USD"",""S"":""BUY"",""tT"":""TRADE"",""T"":1782762030214,""p"":""60264"",""q"":""0.00080""}],""ts"":1782762030221,""seq"":3}","{'data': [{'i': 2071680224830951424, 'm': 'BTC-USD', 'S': 'BUY', 'tT': 'TRADE', 'T': 1782762030214, 'p': '60264', 'q': '0.00080'}], 'ts': 1782762030221, 'se..."



extended/btcspot-usd/trades  0.001 MB


,received_at,received_mts,exchange,symbol,channel,connection_id,payload_text,payload
0,2026-06-29T21:06:44.439Z,1782767204439,extended,BTCSPOT-USD,trades,extended-BTCSPOT-USD-trades,"{""data"":[{""i"":2071579666916839424,""m"":""BTCSPOT-USD"",""S"":""BUY"",""tT"":""TRADE"",""T"":1782738055339,""p"":""59939"",""q"":""0.00041000""},{""i"":2071579800169877504,""m"":""BTC...","{'data': [{'i': 2071579666916839424, 'm': 'BTCSPOT-USD', 'S': 'BUY', 'tT': 'TRADE', 'T': 1782738055339, 'p': '59939', 'q': '0.00041000'}, {'i': 207157980016..."
1,2026-06-29T21:24:34.588Z,1782768274588,extended,BTCSPOT-USD,trades,extended-BTCSPOT-USD-trades,"{""data"":[{""i"":2071706414677495810,""m"":""BTCSPOT-USD"",""S"":""SELL"",""tT"":""TRADE"",""T"":1782768274360,""p"":""60210"",""q"":""0.02072000""}],""ts"":1782768274366,""seq"":2}","{'data': [{'i': 2071706414677495810, 'm': 'BTCSPOT-USD', 'S': 'SELL', 'tT': 'TRADE', 'T': 1782768274360, 'p': '60210', 'q': '0.02072000'}], 'ts': 1782768274..."
2,2026-06-29T21:36:49.588Z,1782769009588,extended,BTCSPOT-USD,trades,extended-BTCSPOT-USD-trades,"{""data"":[{""i"":2071709497490935808,""m"":""BTCSPOT-USD"",""S"":""SELL"",""tT"":""TRADE"",""T"":1782769009360,""p"":""60313"",""q"":""0.40000000""}],""ts"":1782769009365,""seq"":3}","{'data': [{'i': 2071709497490935808, 'm': 'BTCSPOT-USD', 'S': 'SELL', 'tT': 'TRADE', 'T': 1782769009360, 'p': '60313', 'q': '0.40000000'}], 'ts': 1782769009..."



extended/btcspot-usd/orderbook  0.015 MB


,received_at,received_mts,exchange,symbol,channel,connection_id,payload_text,payload
0,2026-06-29T21:06:44.418Z,1782767204418,extended,BTCSPOT-USD,orderbook,extended-BTCSPOT-USD-orderbook,"{""type"":""SNAPSHOT"",""data"":{""t"":""SNAPSHOT"",""m"":""BTCSPOT-USD"",""b"":[{""q"":""0.81559"",""p"":""60154""},{""q"":""0.00083"",""p"":""60150""},{""q"":""0.81559"",""p"":""60128""},{""q"":""0...","{'type': 'SNAPSHOT', 'data': {'t': 'SNAPSHOT', 'm': 'BTCSPOT-USD', 'b': [{'q': '0.81559', 'p': '60154'}, {'q': '0.00083', 'p': '60150'}, {'q': '0.81559', 'p..."
1,2026-06-29T21:06:44.419Z,1782767204419,extended,BTCSPOT-USD,orderbook,extended-BTCSPOT-USD-orderbook,"{""type"":""DELTA"",""data"":{""t"":""DELTA"",""m"":""BTCSPOT-USD"",""b"":[{""q"":""0.81559"",""p"":""60066"",""c"":""0.81559""},{""q"":""-0.81559"",""p"":""60065"",""c"":""0""},{""q"":""-0.81559"",""p...","{'type': 'DELTA', 'data': {'t': 'DELTA', 'm': 'BTCSPOT-USD', 'b': [{'q': '0.81559', 'p': '60066', 'c': '0.81559'}, {'q': '-0.81559', 'p': '60065', 'c': '0'}..."
2,2026-06-29T21:06:44.419Z,1782767204419,extended,BTCSPOT-USD,orderbook,extended-BTCSPOT-USD-orderbook,"{""type"":""DELTA"",""data"":{""t"":""DELTA"",""m"":""BTCSPOT-USD"",""b"":[{""q"":""-0.81559"",""p"":""60009"",""c"":""0""},{""q"":""0.81559"",""p"":""60003"",""c"":""0.81559""}],""a"":[{""q"":""-0.815...","{'type': 'DELTA', 'data': {'t': 'DELTA', 'm': 'BTCSPOT-USD', 'b': [{'q': '-0.81559', 'p': '60009', 'c': '0'}, {'q': '0.81559', 'p': '60003', 'c': '0.81559'}..."


In [5]:
# Quick filter examples
extended_spot = recent.query("exchange == 'extended' and symbol == 'btcspot-usd'")
hyperliquid = recent.query("exchange == 'hyperliquid'")
deribit = recent.query("exchange == 'deribit'")
bitfinex = recent.query("exchange == 'bitfinex'")

display(extended_spot)
display(hyperliquid)
display(deribit)
display(bitfinex)

,exchange,symbol,channel,modified_at,size_mb,path
8,extended,btcspot-usd,trades,2026-06-29 21:46:39.829576+00:00,0.001279,/mnt/burner-archive/ws_raw/extended/btcspot-usd/trades/btcspot-usd_trades_26-06-29.jsonl.zst
9,extended,btcspot-usd,orderbook,2026-06-29 21:46:39.829576+00:00,0.015449,/mnt/burner-archive/ws_raw/extended/btcspot-usd/orderbook/btcspot-usd_orderbook_26-06-29.jsonl.zst


,exchange,symbol,channel,modified_at,size_mb,path
5,hyperliquid,ubtc_usdc,trades,2026-06-29 22:02:47.335338+00:00,0.034681,/mnt/burner-archive/ws_raw/hyperliquid/ubtc_usdc/trades/ubtc_usdc_trades_26-06-29.jsonl.zst
6,hyperliquid,ubtc_usdc,book,2026-06-29 22:02:40.971946+00:00,0.085130,/mnt/burner-archive/ws_raw/hyperliquid/ubtc_usdc/book/ubtc_usdc_book_26-06-29.jsonl.zst
12,hyperliquid,ubtc_usdc,control,2026-06-29 21:46:39.829576+00:00,0.000266,/mnt/burner-archive/ws_raw/hyperliquid/ubtc_usdc/control/ubtc_usdc_control_26-06-29.jsonl.zst


,exchange,symbol,channel,modified_at,size_mb,path
1,bitfinex,tbtcusd,book_l25,2026-06-29 22:06:45.457546+00:00,1.913250,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-29.jsonl.zst
10,bitfinex,tbtcusd,trades,2026-06-29 21:46:39.829576+00:00,0.071091,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-29.jsonl.zst
15,bitfinex,tbtcusd,book_l25,2026-06-23 18:01:54.169195+00:00,16.211049,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-23.jsonl.zst
17,bitfinex,tbtcusd,trades,2026-06-23 18:00:16.614711+00:00,0.838464,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-23.jsonl.zst
23,bitfinex,tbtcusd,trades,2026-06-23 00:00:01.247219+00:00,1.312600,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-22.jsonl.zst
25,bitfinex,tbtcusd,book_l25,2026-06-23 00:00:00.577211+00:00,22.081705,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-22.jsonl.zst
31,bitfinex,tbtcusd,trades,2026-06-22 00:00:03.993046+00:00,1.084338,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-21.jsonl.zst
33,bitfinex,tbtcusd,book_l25,2026-06-22 00:00:00.423006+00:00,13.138911,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-21.jsonl.zst
39,bitfinex,tbtcusd,trades,2026-06-21 00:00:10.613387+00:00,0.904545,/mnt/burner-archive/ws_raw/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-20.jsonl.zst


In [ ]:
# Deribit-specific sample. Today this usually shows the subscription control response
# unless a BTC instrument creation/state event has fired.
deribit_all = recent.query("exchange == 'deribit'")
display(deribit_all)

if deribit_all.empty:
    print("No Deribit files found in the recent list.")
else:
    deribit_path = deribit_all.iloc[0]["path"]
    print(deribit_path)
    display(expand_payload_text(read_jsonl_zst_head(deribit_path, SAMPLE_ROWS)))